In [ ]:
## BICUBIC INTERPOLATION
import os

import cv2
import numpy as np
import torch
from natsort import natsorted

import config
from imgproc import imresize
import imgproc
from model import VDSR
from train import calculate_ssim
from skimage.metrics import structural_similarity as compare_ssim
from train import save_comparision_images

# Directories
hr_dir = "C:/Users/hp/Desktop/Image super-resolution/VDSR-PyTorch/data/Testing/Scale 4/Set 5/HR"
lr_dir = "C:/Users/hp/Desktop/Image super-resolution/VDSR-PyTorch/data/Testing/Scale 4/Set 5/LR"
sr_dir = "C:/Users/hp/Desktop/Image super-resolution/VDSR-PyTorch/results/SR_Images/Bicubic/Scale 4/Set 5"
upscale_factor = 4

def main() -> None:

    # Initialize the image evaluation index.
    total_psnr_grayscale = 0.0
    total_psnr_rgb = 0.0
    total_ssim = 0.0
    total_ssim_rgb = 0.0

    # Get a list of test image file names.
    file_names = natsorted(os.listdir(hr_dir))
    # Get the number of test image files.
    total_files = len(file_names)

    for index in range(total_files):
        filename = file_names[index]
        hr_image_path = os.path.join(hr_dir, filename)
        sr_filename = filename.replace("_HR", "_SR_Bicubic")
        sr_image_path = os.path.join(sr_dir, sr_filename)
        lr_filename = filename.replace("_HR", "_LR")
        lr_image_path = os.path.join(lr_dir, lr_filename)

        print(f"Processing `{os.path.abspath(hr_image_path)}`...")
        # Load and normalize high-resolution image
        hr_image = cv2.imread(hr_image_path).astype(np.float32) / 255.0

        # Load and normalize low-resolution image
        lr_image = cv2.imread(lr_image_path).astype(np.float32) / 255.0
    

        # Convert BGR image to YCbCr image
        lr_ycbcr_image = imgproc.bgr2ycbcr(lr_image, use_y_channel=False)
        hr_ycbcr_image = imgproc.bgr2ycbcr(hr_image, use_y_channel=False)

        # Split YCbCr image data
        lr_y_image, lr_cb_image, lr_cr_image = cv2.split(lr_ycbcr_image)
        hr_y_image, hr_cb_image, hr_cr_image = cv2.split(hr_ycbcr_image)
        
        # Perform interpolation
        sr_y_image = imresize(lr_y_image, upscale_factor)

        # Merge Cb and Cr components
        sr_ycbcr_image = cv2.merge([sr_y_image, hr_cb_image, hr_cr_image])


        # Save image
        sr_image = imgproc.ycbcr2bgr(sr_ycbcr_image)
        cv2.imwrite(sr_image_path, sr_image * 255.0)
        
        ## SSIM Calculationa and concatenation
        # Remove singleton channel if present
        if sr_y_image.ndim == 3 and sr_y_image.shape[2] == 1:
            sr_y_image = sr_y_image[:, :, 0]  # Or np.squeeze(sr_y_image, axis=2)

        ssim_val = compare_ssim(sr_y_image, hr_y_image, data_range=1.0)
        total_ssim += ssim_val
        ssim_val_rgb = compare_ssim(sr_image, hr_image, data_range=1.0, channel_axis = -1)
        total_ssim_rgb += ssim_val_rgb

        mse_gray = np.mean((sr_y_image - hr_y_image)**2)
        if mse_gray == 0:
            return float("inf")
        psnr_grayscale = 10 * np.log10(1.0 / mse_gray)
        total_psnr_grayscale += psnr_grayscale

        mse_rgb = np.mean((sr_image - hr_image)**2)
        if mse_rgb == 0:
            return float("inf")
        psnr_rgb = 10 * np.log10(1.0 / mse_rgb)
        total_psnr_rgb += psnr_rgb
        



    print(f"PSNR(Grayscale): {total_psnr_grayscale / total_files:4.2f}dB.\n")
    print(f"PSNR(RGB): {total_psnr_rgb / total_files:4.2f}dB.\n")
    print(f"Average SSIM(Grayscale): {total_ssim / total_files:4.2f}")
    print(f"Average SSIM(RGB): {total_ssim_rgb / total_files:4.2f}")


if __name__ == "__main__":
    main()


Processing `C:\Users\hp\Desktop\Image super-resolution\VDSR-PyTorch\data\Testing\Scale 4\Set 5\HR\img_001_SRF_4_HR.png`...
Processing `C:\Users\hp\Desktop\Image super-resolution\VDSR-PyTorch\data\Testing\Scale 4\Set 5\HR\img_002_SRF_4_HR.png`...
Processing `C:\Users\hp\Desktop\Image super-resolution\VDSR-PyTorch\data\Testing\Scale 4\Set 5\HR\img_003_SRF_4_HR.png`...
Processing `C:\Users\hp\Desktop\Image super-resolution\VDSR-PyTorch\data\Testing\Scale 4\Set 5\HR\img_004_SRF_4_HR.png`...
Processing `C:\Users\hp\Desktop\Image super-resolution\VDSR-PyTorch\data\Testing\Scale 4\Set 5\HR\img_005_SRF_4_HR.png`...
PSNR(Grayscale): 28.44dB.

PSNR(RGB): 27.11dB.

Average SSIM(Grayscale): 0.82
Average SSIM(RGB): 0.81


In [ ]:
## NEAREST NEIGHBOUR INTERPOLATION
sr_dir = "C:/Users/hp/Desktop/Image super-resolution/VDSR-PyTorch/results/SR_Images/Nearest Neighbour/Scale 4/Set 5"
def main() -> None:

    # Initialize the image evaluation index.
    total_psnr_grayscale = 0.0
    total_psnr_rgb = 0.0
    total_ssim = 0.0
    total_ssim_rgb = 0.0

    # Get a list of test image file names.
    file_names = natsorted(os.listdir(hr_dir))
    # Get the number of test image files.
    total_files = len(file_names)

    for index in range(total_files):
        filename = file_names[index]
        hr_image_path = os.path.join(hr_dir, filename)
        sr_filename = filename.replace("_HR", "_SR_NN")
        sr_image_path = os.path.join(sr_dir, sr_filename)
        lr_filename = filename.replace("_HR", "_LR")
        lr_image_path = os.path.join(lr_dir, lr_filename)

        print(f"Processing `{os.path.abspath(hr_image_path)}`...")        
        # Load and normalize high-resolution image
        hr_image = cv2.imread(hr_image_path).astype(np.float32) / 255.0

        # Load and normalize low-resolution image
        lr_image = cv2.imread(lr_image_path).astype(np.float32) / 255.0

        new_size = (lr_image.shape[1] * upscale_factor, lr_image.shape[0] * upscale_factor)

        
    

        # Convert BGR image to YCbCr image
        lr_ycbcr_image = imgproc.bgr2ycbcr(lr_image, use_y_channel=False)
        hr_ycbcr_image = imgproc.bgr2ycbcr(hr_image, use_y_channel=False)

        # Split YCbCr image data
        lr_y_image, lr_cb_image, lr_cr_image = cv2.split(lr_ycbcr_image)
        hr_y_image, hr_cb_image, hr_cr_image = cv2.split(hr_ycbcr_image)

        # Perform interpolation
        sr_y_image = cv2.resize(lr_y_image, new_size, interpolation = cv2.INTER_NEAREST)

        # Merge Cb and Cr components 
        sr_ycbcr_image = cv2.merge([sr_y_image, hr_cb_image, hr_cr_image])


        # Save image
        sr_image = imgproc.ycbcr2bgr(sr_ycbcr_image)
        cv2.imwrite(sr_image_path, sr_image * 255.0)
        
        ## SSIM Calculationa and concatenation
        # Remove singleton channel if present
        if sr_y_image.ndim == 3 and sr_y_image.shape[2] == 1:
            sr_y_image = sr_y_image[:, :, 0]  # Or np.squeeze(sr_y_image, axis=2)

        ssim_val = compare_ssim(sr_y_image, hr_y_image, data_range=1.0)
        total_ssim += ssim_val
        ssim_val_rgb = compare_ssim(sr_image, hr_image, data_range=1.0, channel_axis = -1)
        total_ssim_rgb += ssim_val_rgb

        mse_gray = np.mean((sr_y_image - hr_y_image)**2)
        if mse_gray == 0:
            return float("inf")
        psnr_grayscale = 10 * np.log10(1.0 / mse_gray)
        total_psnr_grayscale += psnr_grayscale

        mse_rgb = np.mean((sr_image - hr_image)**2)
        if mse_rgb == 0:
            return float("inf")
        psnr_rgb = 10 * np.log10(1.0 / mse_rgb)
        total_psnr_rgb += psnr_rgb

    

    print(f"PSNR(Grayscale): {total_psnr_grayscale / total_files:4.2f}dB.\n")
    print(f"PSNR(RGB): {total_psnr_rgb / total_files:4.2f}dB.\n")
    print(f"Average SSIM(Grayscale): {total_ssim / total_files:4.2f}")
    print(f"Average SSIM(RGB): {total_ssim_rgb / total_files:4.2f}")


if __name__ == "__main__":
    main()


Processing `C:\Users\hp\Desktop\Image super-resolution\VDSR-PyTorch\data\Testing\Scale 4\Set 5\HR\img_001_SRF_4_HR.png`...
Processing `C:\Users\hp\Desktop\Image super-resolution\VDSR-PyTorch\data\Testing\Scale 4\Set 5\HR\img_002_SRF_4_HR.png`...
Processing `C:\Users\hp\Desktop\Image super-resolution\VDSR-PyTorch\data\Testing\Scale 4\Set 5\HR\img_003_SRF_4_HR.png`...
Processing `C:\Users\hp\Desktop\Image super-resolution\VDSR-PyTorch\data\Testing\Scale 4\Set 5\HR\img_004_SRF_4_HR.png`...
Processing `C:\Users\hp\Desktop\Image super-resolution\VDSR-PyTorch\data\Testing\Scale 4\Set 5\HR\img_005_SRF_4_HR.png`...
PSNR(Grayscale): 26.31dB.

PSNR(RGB): 24.99dB.

Average SSIM(Grayscale): 0.76
Average SSIM(RGB): 0.75


In [ ]:
## LANCZOS INTERPOLATION
sr_dir = "C:/Users/hp/Desktop/Image super-resolution/VDSR-PyTorch/results/SR_Images/Lanczos/Scale 4/Set 5"

def main() -> None:

    # Initialize the image evaluation index.
    total_psnr_grayscale = 0.0
    total_psnr_rgb = 0.0
    total_ssim = 0.0
    total_ssim_rgb = 0.0

    # Get a list of test image file names.
    file_names = natsorted(os.listdir(hr_dir))
    # Get the number of test image files.
    total_files = len(file_names)

    for index in range(total_files):
        filename = file_names[index]
        hr_image_path = os.path.join(hr_dir, filename)
        sr_filename = filename.replace("_HR", "_SR_Lanczos")
        sr_image_path = os.path.join(sr_dir, sr_filename)
        lr_filename = filename.replace("_HR", "_LR")
        lr_image_path = os.path.join(lr_dir, lr_filename)

        print(f"Processing `{os.path.abspath(hr_image_path)}`...")
        # Load and normalize high-resolution image
        hr_image = cv2.imread(hr_image_path).astype(np.float32) / 255.0

        # Load and normalize low-resolution image
        lr_image = cv2.imread(lr_image_path).astype(np.float32) / 255.0

        new_size = (lr_image.shape[1] * upscale_factor, lr_image.shape[0] * upscale_factor)

        
    

        # Convert BGR image to YCbCr image
        lr_ycbcr_image = imgproc.bgr2ycbcr(lr_image, use_y_channel=False)
        hr_ycbcr_image = imgproc.bgr2ycbcr(hr_image, use_y_channel=False)

        # Split YCbCr image data
        lr_y_image, lr_cb_image, lr_cr_image = cv2.split(lr_ycbcr_image)
        hr_y_image, hr_cb_image, hr_cr_image = cv2.split(hr_ycbcr_image)

        # Perform interpolation   
        sr_y_image = cv2.resize(lr_y_image, new_size, interpolation = cv2.INTER_LANCZOS4)

        # Merge Cb and Cr compoenets
        sr_ycbcr_image = cv2.merge([sr_y_image, hr_cb_image, hr_cr_image])


        # Save image
        sr_image = imgproc.ycbcr2bgr(sr_ycbcr_image)
        cv2.imwrite(sr_image_path, sr_image * 255.0)
        
        ## SSIM Calculationa and concatenation
        # Remove singleton channel if present
        if sr_y_image.ndim == 3 and sr_y_image.shape[2] == 1:
            sr_y_image = sr_y_image[:, :, 0]  # Or np.squeeze(sr_y_image, axis=2)

        ssim_val = compare_ssim(sr_y_image, hr_y_image, data_range=1.0)
        total_ssim += ssim_val
        ssim_val_rgb = compare_ssim(sr_image, hr_image, data_range=1.0, channel_axis = -1)
        total_ssim_rgb += ssim_val_rgb

        mse_gray = np.mean((sr_y_image - hr_y_image)**2)
        if mse_gray == 0:
            return float("inf")
        psnr_grayscale = 10 * np.log10(1.0 / mse_gray)
        total_psnr_grayscale += psnr_grayscale

        mse_rgb = np.mean((sr_image - hr_image)**2)
        if mse_rgb == 0:
            return float("inf")
        psnr_rgb = 10 * np.log10(1.0 / mse_rgb)
        total_psnr_rgb += psnr_rgb

    

    print(f"PSNR(Grayscale): {total_psnr_grayscale / total_files:4.2f}dB.\n")
    print(f"PSNR(RGB): {total_psnr_rgb / total_files:4.2f}dB.\n")
    print(f"Average SSIM(Grayscale): {total_ssim / total_files:4.2f}")
    print(f"Average SSIM(RGB): {total_ssim_rgb / total_files:4.2f}")


if __name__ == "__main__":
    main()


Processing `C:\Users\hp\Desktop\Image super-resolution\VDSR-PyTorch\data\Testing\Scale 4\Set 5\HR\img_001_SRF_4_HR.png`...
Processing `C:\Users\hp\Desktop\Image super-resolution\VDSR-PyTorch\data\Testing\Scale 4\Set 5\HR\img_002_SRF_4_HR.png`...
Processing `C:\Users\hp\Desktop\Image super-resolution\VDSR-PyTorch\data\Testing\Scale 4\Set 5\HR\img_003_SRF_4_HR.png`...
Processing `C:\Users\hp\Desktop\Image super-resolution\VDSR-PyTorch\data\Testing\Scale 4\Set 5\HR\img_004_SRF_4_HR.png`...
Processing `C:\Users\hp\Desktop\Image super-resolution\VDSR-PyTorch\data\Testing\Scale 4\Set 5\HR\img_005_SRF_4_HR.png`...
PSNR(Grayscale): 28.88dB.

PSNR(RGB): 27.56dB.

Average SSIM(Grayscale): 0.83
Average SSIM(RGB): 0.82
